# CrossCell Benchmark Environment Check

This notebook verifies that all tools are correctly installed in the benchmark environment.

**Tools tested:**
- CrossCell (Rust CLI)
- Python: anndata, scanpy, easySCFpy
- R: zellkonverter, anndataR, convert2anndata, easySCFr, Seurat


## 1. Python Environment

In [ ]:
import sys
print(f"Python: {sys.version}")

import anndata
print(f"anndata: {anndata.__version__}")

import scanpy
print(f"scanpy: {scanpy.__version__}")

import numpy as np
print(f"numpy: {np.__version__}")

import scipy
print(f"scipy: {scipy.__version__}")

import h5py
print(f"h5py: {h5py.__version__}")

import easySCFpy
print(f"easySCFpy: OK")

print("\n✅ Python environment OK")

## 2. R Environment

In [ ]:
import subprocess

r_script = """
cat(paste("R version:", R.version.string), "\n")

pkgs <- c(
    "Seurat",
    "zellkonverter",
    "anndataR",
    "convert2anndata",
    "easySCFr",
    "Matrix",
    "hdf5r"
)

for (pkg in pkgs) {
    v <- tryCatch(
        as.character(packageVersion(pkg)),
        error = function(e) "NOT FOUND"
    )
    status <- ifelse(v == "NOT FOUND", "❌", "✅")
    cat(sprintf("%s %s: %s\n", status, pkg, v))
}
"""

result = subprocess.run(["Rscript", "-e", r_script], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("[stderr]", result.stderr[:500])

## 3. CrossCell CLI

In [ ]:
import subprocess, shutil

path = shutil.which("crosscell")
print(f"crosscell path: {path}")

result = subprocess.run(["crosscell", "--version"], capture_output=True, text=True)
print(f"version: {result.stdout.strip()}")

result = subprocess.run(["crosscell", "--help"], capture_output=True, text=True)
print(f"\n{result.stdout[:500]}")

print("\n✅ CrossCell CLI OK")

## 4. Data Availability

In [ ]:
import os, glob

data_dir = "/benchmark/data"
test_dir = "/benchmark/test_data"

for d, label in [(data_dir, "Benchmark data"), (test_dir, "Test data")]:
    if os.path.isdir(d):
        rds = glob.glob(os.path.join(d, "**/*.rds"), recursive=True)
        h5ad = glob.glob(os.path.join(d, "**/*.h5ad"), recursive=True)
        print(f"{label} ({d}): {len(rds)} .rds, {len(h5ad)} .h5ad")
    else:
        print(f"⚠️  {label} ({d}): directory not found")

print("\n✅ Data check complete")

## 5. Summary

In [ ]:
import json, datetime

summary = {
    "timestamp": datetime.datetime.now().isoformat(),
    "python": sys.version.split()[0],
    "anndata": anndata.__version__,
    "scanpy": scanpy.__version__,
    "crosscell": subprocess.run(["crosscell", "--version"], capture_output=True, text=True).stdout.strip(),
}

# Get R tool versions
r_ver = subprocess.run(["Rscript", "-e", 'cat(R.version.string)'], capture_output=True, text=True)
summary["R"] = r_ver.stdout.strip()

for pkg in ["Seurat", "zellkonverter", "anndataR", "convert2anndata", "easySCFr"]:
    r = subprocess.run(["Rscript", "-e", f"cat(as.character(packageVersion('{pkg}')))"], capture_output=True, text=True)
    summary[pkg] = r.stdout.strip() if r.returncode == 0 else "N/A"

print(json.dumps(summary, indent=2))
print("\n🎉 All checks passed! Environment is ready for benchmarking.")